# NBA Stats Database
Fetches 2025-26 season stats from NBA.com and saves to a local SQLite database for SQL querying.

## 1. Install Dependencies

In [114]:
from nba_api.stats.endpoints import leaguedashplayerstats, leaguedashteamstats
import pandas as pd
import sqlite3
from nba_api.stats.endpoints import leaguegamelog
import time
import os
import math

## 2. Fetch Player Stats from NBA.com

In [115]:

SEASON = '2025-26'

# --- Player Stats ---
print('Fetching player stats...')
player_stats = leaguedashplayerstats.LeagueDashPlayerStats(season=SEASON)
df_players = pd.DataFrame(player_stats.get_data_frames()[0])
print(f'Players fetched: {len(df_players)}')
df_players.head()

Fetching player stats...
Players fetched: 582


,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,W_PCT,...,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,TEAM_COUNT
0,1630639,A.J. Lawson,A.J.,1610612761,TOR,25.0,24,12,12,0.500,...,125,142,443,438,290,450,284,38,449,1
1,1631260,AJ Green,AJ,1610612749,MIL,26.0,78,31,47,0.397,...,271,540,229,126,496,175,284,38,139,1
2,1642358,AJ Johnson,AJ,1610612742,DAL,21.0,48,9,39,0.188,...,371,158,384,400,394,414,284,38,413,2
3,203932,Aaron Gordon,Aaron,1610612743,DEN,30.0,36,27,9,0.750,...,271,242,143,197,48,247,114,38,241,1
4,1628988,Aaron Holiday,Aaron,1610612745,HOU,29.0,57,38,19,0.667,...,179,274,323,321,99,354,284,38,348,1


In [116]:
# --- Team Stats ---
print('Fetching team stats...')
team_stats = leaguedashteamstats.LeagueDashTeamStats(season=SEASON)
df_teams = pd.DataFrame(team_stats.get_data_frames()[0])
print(f'Teams fetched: {len(df_teams)}')
df_teams.head()

Fetching team stats...
Teams fetched: 30


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,...,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK
0,1610612737,Atlanta Hawks,82,46,36,0.561,3951.0,3575,7541,0.474,...,18,1,10,5,18,17,14,26,6,12
1,1610612738,Boston Celtics,82,56,26,0.683,3946.0,3456,7398,0.467,...,3,27,1,28,11,3,9,27,19,4
2,1610612751,Brooklyn Nets,82,20,62,0.244,3951.0,3071,6933,0.443,...,30,26,29,22,23,23,26,21,30,28
3,1610612766,Charlotte Hornets,82,44,38,0.537,3951.0,3357,7292,0.460,...,5,15,25,29,21,9,10,18,13,8
4,1610612741,Chicago Bulls,82,31,51,0.378,3951.0,3476,7417,0.469,...,9,7,23,23,11,26,8,24,12,22


## 3. Save to SQLite Database

In [117]:
from nba_api.stats.endpoints import playerindex
import time

# --- Fetch missing player bio ---
print('Fetching player bio...')
time.sleep(1)
bio_raw = playerindex.PlayerIndex(season='2025-26')
df_bio = pd.DataFrame(bio_raw.get_data_frames()[0])

def height_to_inches(h):
    try:
        ft, inch = str(h).split('-')
        return int(ft) * 12 + int(inch)
    except:
        return None

df_bio['PLAYER_NAME'] = df_bio['PLAYER_FIRST_NAME'] + ' ' + df_bio['PLAYER_LAST_NAME']
df_bio['HEIGHT_INCHES'] = df_bio['HEIGHT'].apply(height_to_inches)
df_bio['WEIGHT'] = pd.to_numeric(df_bio['WEIGHT'], errors='coerce')

conn = sqlite3.connect('nba_stats.db')
df_bio.to_sql('player_bio', conn, if_exists='replace', index=False)
print(f'Bio saved: {len(df_bio)} players')

# --- Rebuild views with correct join column (TEAM_ID) ---
views = {
    'players_with_team': """
        SELECT p.PLAYER_NAME, p.TEAM_ABBREVIATION, p.GP,
               p.PTS, p.REB, p.AST, p.STL, p.BLK, p.FG_PCT, p.FG3_PCT, p.FT_PCT,
               t.W, t.L, t.W_PCT AS TEAM_WIN_PCT
        FROM player_stats p
        JOIN team_stats t ON p.TEAM_ID = t.TEAM_ID
    """,
    'games_enriched': """
        SELECT g.PLAYER_NAME, g.TEAM_ABBREVIATION, g.GAME_DATE, g.MATCHUP,
               g.HOME_AWAY, g.WL, g.PTS, g.REB, g.AST, g.STL, g.BLK,
               g.TOV, g.FG_PCT, g.FG3_PCT, g.FT_PCT, g.MIN,
               p.PTS AS SEASON_AVG_PTS, p.REB AS SEASON_AVG_REB,
               t.W_PCT AS TEAM_WIN_PCT
        FROM game_logs g
        LEFT JOIN player_stats p ON g.PLAYER_NAME = p.PLAYER_NAME
        LEFT JOIN team_stats t   ON g.TEAM_ID = t.TEAM_ID
    """,
    'players_full': """
        SELECT p.*, b.HEIGHT, b.HEIGHT_INCHES, b.WEIGHT,
               b.COUNTRY, b.COLLEGE, b.DRAFT_YEAR, b.DRAFT_NUMBER
        FROM player_stats p
        LEFT JOIN player_bio b ON p.PLAYER_NAME = b.PLAYER_NAME
    """
}

for name, sql in views.items():
    conn.execute(f'DROP VIEW IF EXISTS {name}')
    conn.execute(f'CREATE VIEW {name} AS {sql}')

conn.commit()
conn.close()
print('Done — views rebuilt!')

Fetching player bio...
Bio saved: 587 players
Done — views rebuilt!


In [118]:


# --- Fetch game logs ---
print('Fetching game logs...')
time.sleep(1)
raw = leaguegamelog.LeagueGameLog(season='2025-26', player_or_team_abbreviation='P')
df_games = pd.DataFrame(raw.get_data_frames()[0])
df_games['HOME_AWAY'] = df_games['MATCHUP'].apply(lambda x: 'Home' if 'vs.' in x else 'Away')
print(f'Game logs fetched: {len(df_games)} rows')

# --- Save to DB ---
conn = sqlite3.connect('nba_stats.db')

df_players.to_sql('player_stats', conn, if_exists='replace', index=False)
df_teams.to_sql('team_stats',     conn, if_exists='replace', index=False)
df_games.to_sql('game_logs',      conn, if_exists='replace', index=False)

# --- Views ---
views = {
    'players_with_team': """
        SELECT p.PLAYER_NAME, p.TEAM_ABBREVIATION, p.GP,
               p.PTS, p.REB, p.AST, p.STL, p.BLK, p.FG_PCT, p.FG3_PCT, p.FT_PCT,
               t.W, t.L, t.W_PCT AS TEAM_WIN_PCT
        FROM player_stats p
        JOIN team_stats t ON p.TEAM_ABBREVIATION = t.TEAM_ABBREVIATION
    """,
    'games_enriched': """
        SELECT g.PLAYER_NAME, g.TEAM_ABBREVIATION, g.GAME_DATE, g.MATCHUP,
               g.HOME_AWAY, g.WL, g.PTS, g.REB, g.AST, g.STL, g.BLK,
               g.TOV, g.FG_PCT, g.FG3_PCT, g.FT_PCT, g.MIN,
               p.PTS AS SEASON_AVG_PTS, p.REB AS SEASON_AVG_REB,
               t.W_PCT AS TEAM_WIN_PCT
        FROM game_logs g
        LEFT JOIN player_stats p ON g.PLAYER_NAME = p.PLAYER_NAME
        LEFT JOIN team_stats t   ON g.TEAM_ABBREVIATION = t.TEAM_ABBREVIATION
    """
}

for name, sql in views.items():
    conn.execute(f'DROP VIEW IF EXISTS {name}')
    conn.execute(f'CREATE VIEW {name} AS {sql}')

conn.commit()
conn.close()
print('Database saved as nba_stats.db')

Fetching game logs...
Game logs fetched: 26651 rows
Database saved as nba_stats.db


In [119]:
import ollama

from ollama import chat
from ollama import ChatResponse
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import json


DB_PATH = "nba_stats.db"
model = 'gemma4'

In [120]:
def get_schema():
    conn = sqlite3.connect(DB_PATH)
    schema = conn.execute(
        "SELECT sql FROM sqlite_master WHERE type='table'"
    ).fetchall()
    conn.close()
    return "\n".join([s[0] for s in schema if s[0]])

In [121]:
def run_query(sql, retry=True):
    conn = sqlite3.connect(DB_PATH)
    try:
        df = pd.read_sql(sql, conn)
        conn.close()
        return df
    except Exception as e:
        conn.close()
        if retry:
            fixed_sql = fix_sql(sql, str(e))
            print(f"Fixed SQL: {fixed_sql}")
            return run_query(fixed_sql, retry=False)  # retry once with fix
        else:
            print("Fix agent also failed.")
            raise

In [122]:
import re

def clean_sql(sql):
    # Find AS aliases and wrap ones with hyphens or spaces in backticks
    def fix_alias(match):
        alias = match.group(1)
        if '-' in alias or ' ' in alias:
            return f'AS `{alias}`'
        return f'AS {alias}'

    # Fix AS aliases
    sql = re.sub(r'AS\s+([^\s,\)FROM]+)', fix_alias, sql)

    # Also fix ORDER BY references to the same alias
    sql = re.sub(r'ORDER BY\s+([^\s]+)', lambda m: f'ORDER BY `{m.group(1)}`'
                 if '-' in m.group(1) else f'ORDER BY {m.group(1)}', sql)

    return sql

In [123]:
def fix_sql(bad_sql, error_message):
    print("SQL failed, spawning fix agent...")
    schema = get_schema()
    prompt = f"""You are a SQLite SQL expert helping fix NBA stat queries.

Context: These queries are designed to invent custom basketball statistics by computing
a new column using arithmetic formulas on existing player stat columns (like PTS, REB, AST,
STL, BLK, FG3M, FG3A, FG_PCT etc). The computed column represents a brand new stat and
is used in both the SELECT and ORDER BY clauses.

Database schema:
{schema}

The following SQL query has a syntax error:
SQL: {bad_sql}
Error: {error_message}

SQLite alias rules to follow when fixing:
- Aliases CANNOT start with a number (e.g. 3P_Index → use Three_P_Index)
- Aliases with hyphens MUST use backticks (e.g. Two-Way → `Two-Way`)
- The ORDER BY alias must exactly match the SELECT alias
- Stick to letters, numbers, and underscores in aliases

Return ONLY the corrected SQL query, no explanations, no markdown backticks."""

    response = chat(model='gemma3', messages=[{'role': 'user', 'content': prompt}])
    fixed = response.message.content.strip()

    if fixed.startswith("```"):
        fixed = fixed.split("```")[1]
        if fixed.startswith("sql"):
            fixed = fixed[3:]

    return fixed.strip()

In [124]:
def clean_viz_code(code):
    # Strip markdown fences
    if "```" in code:
        code = code.split("```")[1]
        if code.startswith("python"):
            code = code[6:]

    # Fix malformed plt.show() lines
    lines = code.split("\n")
    cleaned = []
    for line in lines:
        if "plt.show()" in line:
            cleaned.append("plt.show()")  # replace the whole line cleanly
        else:
            cleaned.append(line)

    return "\n".join(cleaned).strip()

In [125]:
# ask another agent
def get_visualization_code(df, stat_name, stat_description):
    prompt = f"""You are a data visualization expert.

I have this pandas DataFrame called 'df' with columns: {df.columns.tolist()}
Here is a sample of the data:
{df.head(10).to_string()}

The stat being visualized is: {stat_name} — {stat_description}

Write Python matplotlib code to visualize this. Follow these rules strictly:
1. Use plt.figure(figsize=(14, 7))
2. Add a main title with the stat name
3. Add a subtitle using plt.suptitle or fig.text showing the formula/description: "{stat_description}"
4. Rotate x-axis labels 45 degrees with ha='right' to avoid overlap
5. Use plt.subplots_adjust(bottom=0.25) to give x labels room
6. Add value labels on top of each bar
7. Make it look clean with a tight layout
Return ONLY executable Python code, no explanations, no markdown backticks."""

    response = chat(model=model, messages=[{'role': 'user', 'content': prompt}])
    return response.message.content.strip()

In [126]:
import ephem
import sqlite3
import pandas as pd
from datetime import datetime

In [127]:

DB_PATH = "nba_stats.db"


def get_moon_phase(date: str) -> dict:
    """
    Get the moon phase for a given game date.
    Args:
        date: Game date in YYYY-MM-DD or YYYY-MM-DD format
    Returns:
        Dictionary with moon_pct (0-100) and moon_label (New Moon, Crescent, Quarter, Gibbous, Full Moon)
    """
    try:
        d = ephem.Date(date.replace('-', '/'))
        pct = round(ephem.Moon(d).phase, 1)

        if pct < 10:   label = 'New Moon'
        elif pct < 35: label = 'Crescent'
        elif pct < 65: label = 'Quarter'
        elif pct < 90: label = 'Gibbous'
        else:          label = 'Full Moon'

        return {"moon_pct": pct, "moon_label": label}
    except Exception as e:
        return {"error": str(e)}


def get_player_game_dates(player_name: str) -> dict:
    """
    Get all game dates for a specific player this season.
    Args:
        player_name: Full name of the NBA player (e.g. 'LeBron James')
    Returns:
        Dictionary with a list of game dates and basic stats per game
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql("""
            SELECT GAME_DATE, MATCHUP, HOME_AWAY, WL, PTS, REB, AST
            FROM game_logs
            WHERE PLAYER_NAME = ?
            ORDER BY GAME_DATE
        """, conn, params=[player_name])
        conn.close()
        return {"games": df.to_dict(orient='records')}
    except Exception as e:
        return {"error": str(e)}


def get_available_tables() -> dict:
    """
    List all tables and views in the NBA database with their column names.
    Returns:
        Dictionary with table/view names and their columns
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        tables = conn.execute(
            "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name"
        ).fetchall()

        result = {}
        for name, kind in tables:
            cols = [row[1] for row in conn.execute(f"PRAGMA table_info({name})").fetchall()]
            result[name] = {"type": kind, "columns": cols}

        conn.close()
        return result
    except Exception as e:
        return {"error": str(e)}


def run_sql_query(sql: str) -> dict:
    """
    Run a SQL query against the NBA database and return the results.
    Args:
        sql: A valid SQLite SELECT query
    Returns:
        Dictionary with the query results as a list of records
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql(sql, conn)
        conn.close()
        return {"results": df.to_dict(orient='records'), "row_count": len(df)}
    except Exception as e:
        return {"error": str(e)}


def get_player_bio(player_name: str) -> dict:
    """
    Get physical and biographical information for a player.
    Args:
        player_name: Full name of the NBA player (e.g. 'Stephen Curry')
    Returns:
        Dictionary with height, weight, college, country, draft info
    """
    try:
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql("""
            SELECT * FROM player_bio WHERE PLAYER_NAME = ?
        """, conn, params=[player_name])
        conn.close()
        if df.empty:
            return {"error": f"No bio found for {player_name}"}
        return df.iloc[0].to_dict()
    except Exception as e:
        return {"error": str(e)}



In [128]:
# ── Dynamic Tools ──────────────────────────────────────────────────────────────
DYNAMIC_TOOLS = True          # Toggle: set False to disable AI tool creation
DYNAMIC_TOOLS_FILE = "dynamic_tools.json"
_dynamic_tools = {}           # name → callable, populated by load/create


def load_dynamic_tools():
    global _dynamic_tools
    if not os.path.exists(DYNAMIC_TOOLS_FILE):
        print("No saved dynamic tools found.")
        return
    with open(DYNAMIC_TOOLS_FILE, 'r') as f:
        saved = json.load(f)
    safe_ns = {
        "sqlite3": sqlite3, "pd": pd, "DB_PATH": DB_PATH,
        "ephem": ephem, "json": json, "math": math, "os": os,
    }
    print(f"Loading {len(saved)} saved dynamic tool(s)...")
    for name, entry in saved.items():
        try:
            exec(entry["code"], safe_ns)
            fn = safe_ns[name]
            fn.__doc__ = entry["description"]
            _dynamic_tools[name] = fn
            print(f"  [OK] {name}")
        except Exception as e:
            print(f"  [Fail] {name}: {e}")


def _save_dynamic_tool(name, description, code, reason):
    saved = {}
    if os.path.exists(DYNAMIC_TOOLS_FILE):
        with open(DYNAMIC_TOOLS_FILE, 'r') as f:
            saved = json.load(f)
    saved[name] = {"description": description, "code": code, "reason": reason}
    with open(DYNAMIC_TOOLS_FILE, 'w') as f:
        json.dump(saved, f, indent=2)


def create_tool(name: str, description: str, python_code: str, reason: str) -> dict:
    """
    Create a new Python tool available for this and future sessions.
    Args:
        name: Function name in snake_case (no spaces or hyphens)
        description: What this tool does (used as its docstring)
        python_code: Complete Python function definition as a string
        reason: Why this tool is being created
    Returns:
        Dictionary with status and tool name
    """
    print(f"\n[New Tool] {name}")
    print(f"  Why:  {reason}")
    print(f"  Does: {description}")

    safe_ns = {
        "sqlite3": sqlite3, "pd": pd, "DB_PATH": DB_PATH,
        "ephem": ephem, "json": json, "math": math, "os": os,
    }
    try:
        exec(python_code, safe_ns)
        fn = safe_ns.get(name)
        if fn is None:
            return {"error": f"Function '{name}' not found in provided code"}
        fn.__doc__ = description
        _dynamic_tools[name] = fn
        _save_dynamic_tool(name, description, python_code, reason)
        print(f"  [OK] Registered and saved to {DYNAMIC_TOOLS_FILE}")
        return {"status": "registered", "tool": name}
    except Exception as e:
        print(f"  [Error] {e}")
        return {"error": str(e)}


load_dynamic_tools()

No saved dynamic tools found.


In [129]:
def assemble_tools() -> list:
    base = [
        get_moon_phase,
        get_player_game_dates,
        get_available_tables,
        run_sql_query,
        get_player_bio,
    ]
    if DYNAMIC_TOOLS:
        base += list(_dynamic_tools.values())
        base.append(create_tool)
    return base

In [130]:
def run_agentic_loop(messages):
    """Run the model in a tool-call loop until it returns a final text answer."""
    while True:
        tools = assemble_tools()
        tool_registry = {fn.__name__: fn for fn in tools}

        response = chat(
            model=model,
            messages=messages,
            tools=tools,
            options={"temperature": 0.1}
        )

        if not response.message.tool_calls:
            return response.message.content

        messages.append(response.message)

        for tc in response.message.tool_calls:
            fn_name = tc.function.name
            fn_args = tc.function.arguments or {}
            fn = tool_registry.get(fn_name)
            if fn is None:
                result = {"error": f"Unknown tool: {fn_name}"}
            else:
                try:
                    result = fn(**fn_args)
                except Exception as e:
                    result = {"error": str(e)}

            messages.append({
                'role': 'tool',
                'content': json.dumps(result),
            })

In [131]:

def invent_stat_and_query(user_question):
    schema = get_schema()
    prompt = f"""You are an NBA statistician with access to tools and this SQLite database schema:

{schema}

The user wants to know: "{user_question}"

Your job:
1. Optionally use tools to explore the database or test queries
2. If you need a capability that no existing tool covers, use create_tool to build it
3. Invent a meaningful custom statistic as a formula using available columns
4. Return ONLY a JSON object with no markdown fences in this exact format:

{{
    "stat_name": "NAME_OF_YOUR_STAT",
    "stat_description": "Explanation of what the stat measures",
    "sql": "SELECT PLAYER_NAME, TEAM_ABBREVIATION, ROUND(formula, 2) AS stat_name FROM player_stats WHERE GP >= 20 ORDER BY stat_name DESC LIMIT 15"
}}"""

    messages = [{'role': 'user', 'content': prompt}]
    raw = run_agentic_loop(messages)

    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    return json.loads(raw.strip())


In [132]:
# response = chat(
#     model    = 'gemma3',
#     messages = [...],
#     stream   = False,
#     options  = {...},
#     keep_alive = '5m',
# ) This is the full chat layout, options is a dict, here is what followsv

# options = {
#     # Creativity — most important setting
#     "temperature": 0.2,   # 0.0 = deterministic/factual, 1.0 = creative/random
#                           # For SQL generation, use 0.1–0.2 for consistency
#
#     # Response length
#     "num_predict": 500,   # max tokens in response — lower = faster
#
#     # Context window
#     "num_ctx": 4096,      # how much of the conversation the model remembers
#
#     # Repetition control
#     "repeat_penalty": 1.1,  # penalizes repeating words, default 1.1
#
#     # Sampling
#     "top_k": 40,          # limits vocab choices per token
#     "top_p": 0.9,         # nucleus sampling threshold
#
#     # Reproducibility
#     "seed": 42,           # set a seed for identical outputs every run
# }

In [133]:
# --- Main pipeline ---
def analyze(question):
    print(f"\nQuestion: {question}")
    print("Inventing stat and generating SQL...")

    result = invent_stat_and_query(question)
    print(f"\nStat: {result['stat_name']}")
    print(f"Description: {result['stat_description']}")
    print(f"SQL: {result['sql']}")

    df = run_query(result['sql'])
    print(f"\nResults ({len(df)} rows):")
    print(df.head())

    print("\nGenerating visualization...")
    viz_code = get_visualization_code(df, result['stat_name'], result['stat_description'])
    viz_code = clean_viz_code(viz_code)  # add this line

    try:
        exec(viz_code, {"df": df, "plt": plt, "pd": pd})
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Visualization error: {e}")
        print("Generated code was:")
        print(viz_code)  # print so you can debug if it fails again



In [134]:
import ipywidgets as widgets
from IPython.display import display, clear_output

In [135]:


question_input = widgets.Text(
    placeholder='e.g. Who is the best two-way player?',
    description='NBA Stat:',
    layout=widgets.Layout(width='500px')
)

submit_btn = widgets.Button(
    description='Analyze',
    button_style='primary',
    layout=widgets.Layout(width='100px')
)

output = widgets.Output()

def on_submit(b):
    with output:
        clear_output(wait=True)
        analyze(question_input.value)

submit_btn.on_click(on_submit)

display(widgets.HBox([question_input, submit_btn]), output)

Output()

In [136]:
analyze(input("stuff"))

KeyboardInterrupt: Interrupted by user